In [80]:
from typing import override
from torch import Tensor, nn
import torch
import numpy as np


# Network hyperparam
dim = 65
L = 4
mode = 8
width = 32
lr = 5e-4
du = 3  # 2 for velociy x, y and pressure.

NU_LAYER = 4
BCU0_LAYER = 5 
BCU1_LAYER = 6
da = du + 3

# NN
# Fourier Convolution Operator
class FCO(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()
        self.mode: int = mode
        
        self.R = nn.Parameter(torch.randn(mode, width, width, dtype=torch.complex64))

    @override
    def forward(self, x: Tensor) -> Tensor:
        # x of shape: (B, dim, dim, width)
        shape = x.shape
        x_hat = torch.fft.fftn(x, dim=(1, 2))[:, :self.mode, :self.mode, :]
        x_hat = torch.einsum('mij,bmni->bmnj', self.R, x_hat)
        x_hat = torch.fft.ifftn(x_hat, s=shape)
        x = x_hat.real
        print(x.shape)
        return x


class PINOLayer(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()

        self.k: nn.Module = FCO(mode, width)
        self.w: nn.Module = nn.Linear(width, width)
        self.sigma: nn.Module = nn.GELU()

    @override
    def forward(self, x: Tensor) -> Tensor:
        w = self.w(x)
        k = self.k(x)
        x = self.sigma(w + k)
        return x


class PINO(nn.Module):
    def __init__(self, da: int, du: int, mode: int, width: int, L: int) -> None:
        super().__init__()

        # uplift
        R = nn.Linear(da, width)
        # projection
        Q = nn.Linear(width, du)

        layers = [PINOLayer(mode, width) for _ in range(L)]
        self.core: nn.Module = nn.Sequential(
            R,
            *layers,
            Q,
        )

    @override
    def forward(self, x: Tensor) -> Tensor:
        return self.core(x)




# PDE param
T = (5, 10)
dimT = 50
alpha = 1
beta = 1

# re & nu not fixed

k = 1
dt = 1e-4


# Todo: fill in the choosen times
t = torch.Tensor()
x = torch.Tensor()
y = torch.Tensor()

# Training is done over elements of shape:
# batch, dimT, dim, dim, C
# We have to compute dimT elements before applying loss because pde loss needs it
# We could apply data loss more often but it would be a different scheme than L = Ldata + λLpde
# The batch dimension mixes data from different 

# To compute pde loss, we need pde params
# Dataset of shape a -> u
# Where a: (batch, dimT, dim, dim, C) with C = da (ex: du + broadcasted pde_params)
# u: (batch, dimT, dim, dim, C) with C = du

L2 = torch.nn.MSELoss(reduction="mean")


# a of shape (batch, dimT, dim, dim, da)
# PDE param embedded in da
# u  of shape (batch, dimT, dim, dim, du)

# residual loss over time
def residual_loss(a, u):
    # assumed density to be 1
    # v viscosity (nu)
    # du/dt = -(u.∇)u + v∇.∇u -∇p
    # R = du/dt + (u.∇)u + ∇p - v∇.∇u

    nu = a[:, NU_LAYER]

    u_vel = u[..., :2]
    p = u[..., 2]
    
    du_dt = torch.autograd.grad(
            outputs=u_vel,
            inputs=t,
            grad_outputs=torch.ones_like(u_vel),
            create_graph=True,
        )[0]

    du_dx, du_dy = torch.autograd.grad(
            outputs=u_vel,
            inputs=(x, y),
            grad_outputs=(torch.ones_like(u_vel)),
            create_graph=True
        )
    convection = torch.empty_like(u_vel)
    convection[..., 0] = u_vel[..., 0] * du_dx[..., 0] + u_vel[..., 1] * du_dy[..., 0]
    convection[..., 1] = u_vel[..., 0] * du_dx[..., 1] + u_vel[..., 1] * du_dy[..., 1]
    
    du_dx2 = torch.autograd.grad(du_dx, x, grad_outputs=torch.ones_like(du_dx), create_graph=True)[0]
    du_dy2 = torch.autograd.grad(du_dy, y, grad_outputs=torch.ones_like(du_dx), create_graph=True)[0]

    viscosity = nu * (du_dx2 + du_dy2)
    
    dp_dx, dp_dy = torch.autograd.grad(
            outputs=p,
            inputs=(x, y),
            grad_outputs=torch.ones_like(p),
            create_graph=True
        )
    grad_p = torch.stack((dp_dx, dp_dy), dim=-1)
    
    R = du_dt + convection + grad_p - viscosity

    return torch.mean(R**2)

# boundary condition loss over time
def bc_loss(a, u):
    pass

# initial condition loss
def ic_loss(a, u):
    pass

# L pde is J pde as long as loss is averaged over batch
def L_pde(a, u):
    return residual_loss(a, u) + alpha * bc_loss(a, u) + beta * ic_loss(a, u)



In [90]:
pino = PINO(da, du, mode, width, L)

In [91]:
Batch = 2
test_input = torch.rand(Batch, dim, dim, da)

In [1]:
output = pino(test_input)

NameError: name 'pino' is not defined

In [77]:
loss = output.sum()


In [78]:
loss.backward()

In [79]:
loss

tensor(706.2964, grad_fn=<SumBackward0>)